In [1]:
import pandas as pd
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

URL = ('https://archive.ics.uci.edu/static/public/222/bank+marketing.zip')
outer = ZipFile(BytesIO(urlopen(URL).read()))
inner = ZipFile(BytesIO(outer.read('bank-additional.zip')))
df = pd.read_csv(inner.open('bank-additional/bank-additional.csv'), sep=';')
df.to_csv('bank.csv', sep=';', index=False)
print(df.shape)
df.head()

(4119, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


In [3]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

# Encode categorical features
X = pd.get_dummies(df.drop(columns=['y']), drop_first=True)

# Convert target variable to binary (0 and 1)
y = df['y'].map({'yes': 1, 'no': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = GradientBoostingClassifier(
	n_estimators=100,
	learning_rate=0.1,
	max_depth=3,
	random_state=42
)
model.fit(X_train, y_train)
print("Train Accuracy:", round(model.score(X_train, y_train), 2))
print("Test Accuracy:", round(model.score(X_test, y_test), 2))


Train Accuracy: 0.95
Test Accuracy: 0.91


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ('sc', StandardScaler()),
    ('model', GradientBoostingClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 5, 10],   
    'model__learning_rate': [0.01, 0.1, 0.2]
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best score:", grid.best_score_)
print("Best params:", grid.best_params_)

Best score: 0.9122913505311077
Best params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

pipeline = Pipeline([
    ('sc', StandardScaler()),
    ('model', GradientBoostingClassifier(random_state=42))
])

param_distributions = {
    'model__n_estimators': randint(50, 250),  
    'model__max_depth': randint(1, 6),        
    'model__learning_rate': uniform(0.01, 0.2), 
    'model__subsample': [0.7, 0.8, 0.9, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20,           
    cv=5,   
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Eng yaxshi score:", random_search.best_score_)
print("Eng yaxshi parametrlar:", random_search.best_params_)

c:\Users\Acer\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Eng yaxshi score: 0.9150227617602429
Eng yaxshi parametrlar: {'model__learning_rate': np.float64(0.041198904067240534), 'model__max_depth': 3, 'model__n_estimators': 137, 'model__subsample': 0.7}
